# المدارس المستخدمه
1️⃣ المدارس الرقمية والمؤشرات (Indicators School)
(RSI, MACD, Moving Averages)

2️⃣ المدارس المتقدمة (Advanced Schools)
(Volume analysis, Time analysis,Multi timeframe)


#نوع التحليل المستخدم
#1️⃣Technical Analysis

المؤشرات المستخدمة:
Trend indicators

EMA 12,
EMA 26,
EMA 50,
EMA 200,
SMA 50,
SMA 200

Momentum

RSI,
MACD,
MACD histogram,
Volatility,
ATR,
ATR %,
Price behaviour,
body size,
wick size,
close position
#2️⃣Trend Analysis
trend_filter = ema_50 > ema_200
و:
trend_strength
ema_slope_50
ema_slope_200

ده يسمح للموديل يفهم:
هل السوق Trending,
قوة الاتجاه,
سرعة الاتجاه
#3️⃣Market Structure Analysis
market_structure,
bos_up,
bos_down

الموديل يقدر يفهم:

Higher High,
Lower Low,
Break of Structure
#4️⃣Support & Resistance Analysis
add support
resistance
and
dist_to_support,
dist_to_resistance,
sr_dist
ده يسمح للموديل يفهم:

السعر قريب من دعم,
السعر قريب من مقاومة,
مكان السعر داخل الرينج
#5️⃣Volume Analysis
add vwap, dist_to_vwap, price_vs_vwap
و: OBV,obv, obv_slope
و:
volume_surge, vol_zone_bull, vol_zone_bear
ده يجعل الموديل يفهم:

أين يدخل المال في السوق,
هل الحركة مدعومة بحجم

#6️⃣Session Analysis

add session of trading such as:

Asia, London, NewYork ,Overlap

#Import libraries

In [ ]:
import requests
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score
from sklearn.model_selection import TimeSeriesSplit
import joblib
import os
import json
from datetime import datetime, timedelta
import time
import logging
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)

In [ ]:
!pip uninstall xgboost
!pip install xgboost==1.7.6

Found existing installation: xgboost 1.7.6
Uninstalling xgboost-1.7.6:
  Would remove:
    /usr/local/lib/python3.12/dist-packages/xgboost-1.7.6.dist-info/*
    /usr/local/lib/python3.12/dist-packages/xgboost.libs/libgomp-a34b3233.so.1.0.0
    /usr/local/lib/python3.12/dist-packages/xgboost/*
Proceed (Y/n)? y
  Successfully uninstalled xgboost-1.7.6
  Using cached xgboost-1.7.6-py3-none-manylinux2014_x86_64.whl.metadata (1.9 kB)
Using cached xgboost-1.7.6-py3-none-manylinux2014_x86_64.whl (200.3 MB)


In [ ]:
import xgboost
print(xgboost.__version__)

1.7.6


In [ ]:
#added:
#   ✅ Market Structure  (HH / HL / LH / LL)
#   ✅ Trend Filter      (EMA cross + slope)
#   ✅ Support & Resistance (rolling pivots)
#   ✅ Volume Profile    (VWAP + OBV + volume zones)
#   ✅ Multi-Timeframe   (كل موديل يرى HTF context)
# ============================================================

import requests
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score
from sklearn.model_selection import TimeSeriesSplit
import joblib
import os
import json
from datetime import datetime, timedelta
import time
import logging
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)

# ─────────────────────────────────────────────
TWELVE_DATA_API_KEY = "2f600a5099a04ee0b652c777edb6c8f8"
SYMBOL      = "XAU/USD"
SYMBOL_SAFE = "XAUUSD"
CSV_RAW_FILE = f"{SYMBOL_SAFE}_5min_raw.csv"

# ─────────────────────────────────────────────
# TIMEFRAME CONFIG
# larger_tf   → الـ HTF اللي هيتسحب منه features إضافية
# htf_features → قائمة الـ features المطلوبة من الـ HTF
# ─────────────────────────────────────────────
TIMEFRAME_CONFIG = {
    # ─────────────────────────────────────────────────────────────
    #  5min: الـ folds كانت خسارة بسبب threshold منخفض → صفقات noise
    #    الإصلاح: رفع buy_threshold لـ 0.67 وخفض fold لـ 0.60
    #    + max_trades_per_day=10 لمنع overfitting على noise
    # ─────────────────────────────────────────────────────────────
    "5min": {
        "interval": "5min", "years": 2,
        "buy_threshold": 0.67,       # ارتفع من 0.60 → صفقات أقل وأدق
        "sell_threshold": 0.33,      # انخفض من 0.40
        "fold_buy_threshold": 0.60,  # ارتفع من 0.55
        "fold_sell_threshold": 0.40, # انخفض من 0.45
        "max_trades_per_day": 10,    #  جديد: حد أقصى للصفقات اليومية
        "resample_rule": "5min", "larger_tf": "1h",
        "htf_features": ["rsi", "trend_filter", "macd_hist", "market_structure"],
        "minutes_per_candle": 5, "active_for_live": True, "min_candles": 500
    },
    # ─────────────────────────────────────────────────────────────
    #  15min: نفس مشكلة الـ 5min — win rate 4-10% في الـ folds
    #    الإصلاح: رفع threshold لـ 0.68 لانتقاء صفقات أقل وأجود
    # ─────────────────────────────────────────────────────────────
    "15min": {
        "interval": "15min", "years": 3,
        "buy_threshold": 0.68,       # ارتفع من 0.65
        "sell_threshold": 0.32,      # انخفض من 0.35
        "fold_buy_threshold": 0.61,  # ارتفع من 0.58
        "fold_sell_threshold": 0.39, # انخفض من 0.42
        "max_trades_per_day": 5,     #  جديد
        "resample_rule": "15min", "larger_tf": "4h",
        "htf_features": ["rsi", "trend_filter", "atr_pct", "market_structure"],
        "minutes_per_candle": 15, "active_for_live": True, "min_candles": 500
    },
    # ─────────────────────────────────────────────────────────────
    #  45min: النتائج جيدة (WR 52.5%, PF 23.64) — لا تغيير جوهري
    #    تعديل طفيف: رفع threshold قليلاً لتحسين جودة الصفقات
    # ─────────────────────────────────────────────────────────────
    "45min": {
        "interval": "45min", "years": 3,
        "buy_threshold": 0.66,       # ارتفع طفيفاً من 0.65
        "sell_threshold": 0.34,
        "fold_buy_threshold": 0.58,
        "fold_sell_threshold": 0.42,
        "resample_rule": "45min", "larger_tf": "4h",
        "htf_features": ["rsi", "trend_filter", "macd_hist"],
        "minutes_per_candle": 45, "active_for_live": True, "min_candles": 300
    },
    # ─────────────────────────────────────────────────────────────
    #  1h: ممتاز (PF 49.97, DD -0.04%) — لا تغيير
    # ─────────────────────────────────────────────────────────────
    "1h": {
        "interval": "1h", "years": 4,
        "buy_threshold": 0.70,       # بدون تغيير — النتائج ممتازة
        "sell_threshold": 0.30,
        "fold_buy_threshold": 0.60,
        "fold_sell_threshold": 0.40,
        "resample_rule": "1h", "larger_tf": "4h",
        "htf_features": ["rsi", "trend_filter", "market_structure", "sr_dist"],
        "minutes_per_candle": 60, "active_for_live": True, "min_candles": 500
    },
    # ─────────────────────────────────────────────────────────────
    #  2h: الأفضل (WR 65.6%, PF 73.94, DD -0.02%) — لا تغيير
    # ─────────────────────────────────────────────────────────────
    "2h": {
        "interval": "2h", "years": 4,
        "buy_threshold": 0.70,       # بدون تغيير — النتائج الأفضل
        "sell_threshold": 0.30,
        "fold_buy_threshold": 0.60,
        "fold_sell_threshold": 0.40,
        "resample_rule": "2h", "larger_tf": "1d",
        "htf_features": ["rsi", "trend_filter", "atr_pct"],
        "minutes_per_candle": 120, "active_for_live": True, "min_candles": 300
    },
    # ─────────────────────────────────────────────────────────────
    #  4h: ممتاز (WR 80.95%, Exp 7.75) — لا تغيير
    # ─────────────────────────────────────────────────────────────
    "4h": {
        "interval": "4h", "years": 5,
        "buy_threshold": 0.72,       # بدون تغيير
        "sell_threshold": 0.28,
        "fold_buy_threshold": 0.62,
        "fold_sell_threshold": 0.38,
        "resample_rule": "4h", "larger_tf": "1d",
        "htf_features": ["rsi", "trend_filter", "market_structure", "sr_dist"],
        "minutes_per_candle": 240, "active_for_live": True, "min_candles": 200
    },
    # ─────────────────────────────────────────────────────────────
    #  1d: 9 صفقات فقط في الـ holdout — مشكلة إحصائية
    #    الإصلاح: خفض threshold لـ 0.58 لتوليد صفقات أكثر
    # ─────────────────────────────────────────────────────────────
    "1d": {
        "interval": "1d", "years": 10,
        "buy_threshold": 0.58,       # انخفض من 0.65 → صفقات أكثر
        "sell_threshold": 0.42,      # ارتفع من 0.35
        "fold_buy_threshold": 0.53,  # انخفض من 0.55
        "fold_sell_threshold": 0.47,
        "resample_rule": "1B", "larger_tf": "1W",
        "htf_features": ["rsi", "trend_filter"],
        "minutes_per_candle": 1440, "active_for_live": True, "min_candles": 100
    },
    # ─────────────────────────────────────────────────────────────
    #  1W: 3 صفقات فقط — استخدمه كـ filter فقط لا للتداول المستقل
    #    الإصلاح: خفض threshold لـ 0.55 + تفعيله كـ filter للـ 1d
    # ─────────────────────────────────────────────────────────────
    "1W": {
        "interval": "1week", "years": 15,
        "buy_threshold": 0.55,       # انخفض من 0.65 → صفقات أكثر
        "sell_threshold": 0.45,      # ارتفع من 0.35
        "fold_buy_threshold": 0.52,  # انخفض من 0.55
        "fold_sell_threshold": 0.48,
        "resample_rule": "W-FRI", "larger_tf": None,
        "htf_features": [],
        "minutes_per_candle": 10080, "active_for_live": True, "min_candles": 50
    }
}

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler("training.log"),
        logging.StreamHandler()
    ]
)

# ══════════════════════════════════════════════════════════════
# DATA FETCHER
# ══════════════════════════════════════════════════════════════
class TwelveDataFetcher:
    BASE_URL = "https://api.twelvedata.com/time_series"

    def __init__(self, api_key):
        self.api_key = api_key

    def fetch_range(self, symbol=SYMBOL, interval="5min",
                    start_date=None, end_date=None, outputsize=5000):
        params = {
            'symbol': symbol, 'interval': interval,
            'apikey': self.api_key, 'format': 'JSON',
            'order': 'ASC', 'outputsize': outputsize
        }
        if start_date: params['start_date'] = start_date
        if end_date:   params['end_date']   = end_date

        logging.info(f"جلب {symbol} ({interval}) من {start_date} إلى {end_date} ...")
        try:
            r = requests.get(self.BASE_URL, params=params)
            if r.status_code != 200:
                logging.error(f"خطأ في الاتصال: {r.status_code}"); return pd.DataFrame()
            data = r.json()
            if 'values' not in data:
                logging.error(f"لا بيانات: {data.get('message','')}"); return pd.DataFrame()

            df = pd.DataFrame(data['values'])
            if 'volume' not in df.columns: df['volume'] = 0
            for col in ['open','high','low','close','volume']:
                df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)
            df['datetime'] = pd.to_datetime(df['datetime'])
            df.sort_values('datetime', inplace=True)
            df.reset_index(drop=True, inplace=True)
            logging.info(f"تم جلب {len(df)} شمعة.")
            return df
        except Exception as e:
            logging.error(f"خطأ غير متوقع: {e}"); return pd.DataFrame()


# ══════════════════════════════════════════════════════════════
# DATA MANAGEMENT
# ══════════════════════════════════════════════════════════════
def fetch_full_history(target_candles=100000, interval="5min"):
    fetcher = TwelveDataFetcher(TWELVE_DATA_API_KEY)
    all_data, total_candles = [], 0
    end_date = datetime.utcnow()
    batch_size, previous_oldest = 5000, None
    max_requests, request_count = 20, 0

    while total_candles < target_candles and request_count < max_requests:
        request_count += 1
        start_date = end_date - timedelta(seconds=batch_size * 300)
        df_batch = fetcher.fetch_range(
            symbol=SYMBOL, interval=interval,
            start_date=start_date.strftime("%Y-%m-%d %H:%M:%S"),
            end_date=end_date.strftime("%Y-%m-%d %H:%M:%S"),
            outputsize=batch_size
        )
        if df_batch.empty: break
        all_data.append(df_batch)
        total_candles += len(df_batch)
        logging.info(f"تم تجميع {total_candles} شمعة.")
        oldest = df_batch['datetime'].min()
        if previous_oldest == oldest: break
        previous_oldest = oldest
        end_date = oldest - timedelta(seconds=1)
        time.sleep(0.5)

    if not all_data: return pd.DataFrame()
    df = pd.concat(all_data, ignore_index=True)
    df.drop_duplicates(subset='datetime', keep='first', inplace=True)
    df.sort_values('datetime', inplace=True)
    df.reset_index(drop=True, inplace=True)
    if len(df) > target_candles:
        df = df.iloc[-target_candles:].reset_index(drop=True)
    logging.info(f"إجمالي الشموع: {len(df)}")
    return df


def update_data(csv_path, new_candles=1000, interval="5min"):
    if not os.path.exists(csv_path):
        logging.error(f"الملف {csv_path} غير موجود."); return None
    df_old = pd.read_csv(csv_path)
    df_old['datetime'] = pd.to_datetime(df_old['datetime'])
    last_date = df_old['datetime'].max()
    fetcher = TwelveDataFetcher(TWELVE_DATA_API_KEY)
    df_new = fetcher.fetch_range(
        symbol=SYMBOL, interval=interval,
        start_date=(last_date + timedelta(seconds=1)).strftime("%Y-%m-%d %H:%M:%S"),
        end_date=datetime.utcnow().strftime("%Y-%m-%d %H:%M:%S"),
        outputsize=new_candles
    )
    if df_new.empty:
        logging.warning("لا توجد بيانات جديدة."); return df_old
    df = pd.concat([df_old, df_new], ignore_index=True)
    df.drop_duplicates(subset='datetime', keep='last', inplace=True)
    df.sort_values('datetime', inplace=True)
    df.reset_index(drop=True, inplace=True)
    df.to_csv(csv_path, index=False)
    logging.info(f"تم التحديث. الإجمالي: {len(df)}")
    return df


def get_or_update_data(csv_path, target_candles=100000, force_refresh=False):
    if os.path.exists(csv_path) and not force_refresh:
        df = update_data(csv_path, new_candles=1000)
        if df is not None: return df
        df = pd.read_csv(csv_path)
        df['datetime'] = pd.to_datetime(df['datetime'])
        return df
    df = fetch_full_history(target_candles=target_candles)
    if not df.empty: df.to_csv(csv_path, index=False)
    return df


# ══════════════════════════════════════════════════════════════
# FEATURE ENGINEER  —
# ══════════════════════════════════════════════════════════════
class FeatureEngineer:
    def __init__(self):
        self.scaler = RobustScaler()
        self.feature_columns = None
        self.scaler_fitted = False

    # ──────────────────────────────────────────
    # 1. Time Features
    # ──────────────────────────────────────────
    def add_time_features(self, df):
        df = df.copy()
        df['hour']        = df['datetime'].dt.hour
        df['day_of_week'] = df['datetime'].dt.dayofweek
        df['minute']      = df['datetime'].dt.minute
        df['hour_sin']    = np.sin(2 * np.pi * df['hour'] / 24)
        df['hour_cos']    = np.cos(2 * np.pi * df['hour'] / 24)
        df['minute_sin']  = np.sin(2 * np.pi * df['minute'] / 60)
        df['minute_cos']  = np.cos(2 * np.pi * df['minute'] / 60)
        h = df['hour']
        df['is_asia']     = ((h >= 0)  & (h < 9)).astype(int)
        df['is_london']   = ((h >= 8)  & (h < 17)).astype(int)
        df['is_ny']       = ((h >= 13) & (h < 22)).astype(int)
        df['is_overlap']  = ((h >= 13) & (h < 17)).astype(int)
        df['is_friday']   = (df['day_of_week'] == 4).astype(int)
        return df

    # ──────────────────────────────────────────
    # 2. Base Technical Indicators
    # ──────────────────────────────────────────
    def compute_indicators(self, df):
        df = df.copy()
        c, h, l, o, v = df['close'], df['high'], df['low'], df['open'], df['volume']

        df['ema_12']  = c.ewm(span=12).mean()
        df['ema_26']  = c.ewm(span=26).mean()
        df['ema_50']  = c.ewm(span=50).mean()
        df['ema_200'] = c.ewm(span=200).mean()
        df['sma_50']  = c.rolling(50).mean()
        df['sma_200'] = c.rolling(200).mean()

        df['macd']        = df['ema_12'] - df['ema_26']
        df['macd_signal'] = df['macd'].ewm(span=9).mean()
        df['macd_hist']   = df['macd'] - df['macd_signal']

        delta    = c.diff()
        avg_gain = delta.clip(lower=0).rolling(14).mean()
        avg_loss = (-delta.clip(upper=0)).rolling(14).mean()
        df['rsi'] = 100 - (100 / (1 + avg_gain / (avg_loss + 1e-9)))

        tr = pd.concat([
            h - l,
            (h - c.shift()).abs(),
            (l - c.shift()).abs()
        ], axis=1).max(axis=1)
        df['atr']     = tr.rolling(14).mean()
        df['atr_pct'] = df['atr'] / c

        df['volume_sma']   = v.rolling(20).mean()
        df['volume_ratio'] = v / (df['volume_sma'] + 1e-9)

        df['returns']         = c.pct_change()
        df['volatility']      = df['returns'].rolling(20).std()
        df['high_low_pct']    = (h - l) / c
        df['close_position']  = (c - l) / (h - l + 1e-9)
        df['dist_to_ema50']   = (c - df['ema_50'])  / c
        df['dist_to_ema200']  = (c - df['ema_200']) / c

        df['body_size']      = abs(c - o)
        df['wick_size']      = (h - l) - df['body_size']
        df['body_ratio']     = df['body_size'] / (df['wick_size'] + 1e-9)
        df['close_open_pct'] = (c - o) / (o + 1e-9)

        return df

    # ──────────────────────────────────────────
    # 3.  Trend Filter
    #    trend_filter  : 1=uptrend  0=downtrend
    #    ema_slope_*   : سرعة الاتجاه
    #    trend_strength: قوة الاتجاه النسبية
    # ──────────────────────────────────────────
    def add_trend_filter(self, df):
        df = df.copy()
        df['trend_filter']   = (df['ema_50'] > df['ema_200']).astype(int)
        df['ema_slope_50']   = df['ema_50'].diff(3)  / (df['close'] + 1e-9)
        df['ema_slope_200']  = df['ema_200'].diff(3) / (df['close'] + 1e-9)
        df['price_vs_ema50'] = (df['close'] > df['ema_50']).astype(int)
        df['price_vs_ema200']= (df['close'] > df['ema_200']).astype(int)
        df['trend_strength'] = (df['ema_50'] - df['ema_200']) / (df['close'] + 1e-9)
        return df

    # ──────────────────────────────────────────
    # 4. Market Structure  (HH/HL/LH/LL)
    #    market_structure: 1=Bullish BOS  -1=Bearish BOS  0=neutral
    #    bos_up / bos_down: Break of Structure
    # ──────────────────────────────────────────
    def add_market_structure(self, df, window=10):
        df   = df.copy()
        high = df['high']
        low  = df['low']

        roll_high      = high.rolling(window).max()
        roll_low       = low.rolling(window).min()
        prev_roll_high = roll_high.shift(window)
        prev_roll_low  = roll_low.shift(window)

        df['hh'] = (roll_high > prev_roll_high).astype(int)
        df['hl'] = (roll_low  > prev_roll_low).astype(int)
        df['lh'] = (roll_high < prev_roll_high).astype(int)
        df['ll'] = (roll_low  < prev_roll_low).astype(int)

        df['market_structure'] = np.where(
            (df['hh'] == 1) & (df['hl'] == 1),   1,
            np.where(
                (df['lh'] == 1) & (df['ll'] == 1), -1, 0
            )
        )
        df['bos_up']   = (df['close'] > roll_high.shift(1)).astype(int)
        df['bos_down'] = (df['close'] < roll_low.shift(1)).astype(int)

        return df

    # ──────────────────────────────────────────
    # 5. Support & Resistance
    #    resistance / support : مستويات S/R بناءً على rolling
    #    dist_to_resistance/support : مسافة نسبية
    #    sr_dist : موضع السعر بين الدعم والمقاومة (0→1)
    # ──────────────────────────────────────────
    def add_support_resistance(self, df, window=20):
        df = df.copy()
        c  = df['close']

        df['resistance'] = df['high'].rolling(window).max()
        df['support']    = df['low'].rolling(window).min()

        df['dist_to_resistance'] = (df['resistance'] - c) / (c + 1e-9)
        df['dist_to_support']    = (c - df['support'])    / (c + 1e-9)
        df['near_resistance']    = (df['dist_to_resistance'] < 0.001).astype(int)
        df['near_support']       = (df['dist_to_support']    < 0.001).astype(int)

        sr_range     = df['resistance'] - df['support']
        df['sr_dist']= (c - df['support']) / (sr_range + 1e-9)

        return df

    # ──────────────────────────────────────────
    # 6.  Volume Profile  (مهم جداً للذهب)
    #    vwap        : Volume Weighted Average Price
    #    dist_to_vwap: المسافة من الـ VWAP
    #    obv         : On-Balance Volume + slope
    #    vol_zone_*  : حجم صاعد/هابط نسبي
    #    volume_surge: شمعة بحجم استثنائي
    # ──────────────────────────────────────────
    def add_volume_profile(self, df):
        df = df.copy()
        c, h, l, v = df['close'], df['high'], df['low'], df['volume']

        typical_price       = (h + l + c) / 3
        df['vwap']          = (typical_price * v).rolling(288).sum() / (v.rolling(288).sum() + 1e-9)
        df['dist_to_vwap']  = (c - df['vwap']) / (df['vwap'] + 1e-9)
        df['price_vs_vwap'] = (c > df['vwap']).astype(int)

        bull_vol = v.where(c > c.shift(1), 0)
        bear_vol = v.where(c < c.shift(1), 0)
        vol_sma  = v.rolling(20).mean()
        df['vol_zone_bull'] = bull_vol.rolling(20).mean() / (vol_sma + 1e-9)
        df['vol_zone_bear'] = bear_vol.rolling(20).mean() / (vol_sma + 1e-9)

        obv_dir      = np.sign(c.diff())
        df['obv']    = (v * obv_dir).cumsum()
        df['obv_sma']= df['obv'].rolling(20).mean()
        df['obv_slope'] = df['obv'].diff(5) / (df['obv'].abs() + 1e-9)

        df['volume_surge'] = (v > vol_sma * 2).astype(int)

        return df

    # ──────────────────────────────────────────
    # MAIN: create_features
    # ──────────────────────────────────────────
    def create_features(self, df, shift=True):
        df = self.add_time_features(df)
        df = self.compute_indicators(df)
        df = self.add_trend_filter(df)
        df = self.add_market_structure(df)
        df = self.add_support_resistance(df)
        df = self.add_volume_profile(df)

        exclude_from_shift = [
            'datetime','open','high','low','close','volume',
            'hour','minute','day_of_week',
            'hour_sin','hour_cos','minute_sin','minute_cos',
            'is_asia','is_london','is_ny','is_overlap','is_friday'
        ]
        tech_cols = [c for c in df.columns if c not in exclude_from_shift]

        if shift:
            for col in tech_cols:
                df[col] = df[col].shift(1)

        # إصلاح dropna — لا نحذف صفوفاً، نحذف أعمدة فاضية فقط
        thresh = int(len(df) * 0.5)
        df = df.dropna(axis=1, thresh=thresh)
        warmup = max(1, len(df) // 10)
        df = df.iloc[warmup:].copy()
        df = df.ffill().fillna(0)

        exclude_final = [
            'datetime','open','high','low','close','volume',
            'future_close','pct_change','target'
        ]
        self.feature_columns = [c for c in df.columns if c not in exclude_final]
        return df

    def create_target(self, df, horizon=1, threshold=None):
        df['future_close'] = df['close'].shift(-horizon)
        df['pct_change']   = (df['future_close'] - df['close']) / df['close']
        if threshold is not None:
            df['target'] = np.where(df['pct_change'] >  threshold, 2,
                           np.where(df['pct_change'] < -threshold, 0, 1))
        else:
            df['target'] = (df['future_close'] > df['close']).astype(int)
        df.dropna(subset=['future_close'], inplace=True)
        df['target'] = df['target'].astype(int)
        return df

    def fit_scaler(self, X):
        self.scaler.fit(X)
        self.scaler_fitted = True
        if isinstance(X, pd.DataFrame):
            self.feature_columns = list(X.columns)

    def transform(self, X):
        if not self.scaler_fitted:
            raise RuntimeError("يجب تدريب الـ scaler أولاً")
        if isinstance(X, pd.DataFrame):
            X = X[self.feature_columns]
        return self.scaler.transform(X)

    def fit_transform(self, X):
        self.fit_scaler(X)
        return self.transform(X)


# ══════════════════════════════════════════════════════════════
#  MULTI-TIMEFRAME FEATURE BUILDER
#    كل موديل LTF يرى context من الـ HTF
#    merge_asof → لا lookahead: كل شمعة LTF تأخذ آخر HTF قبلها
# ══════════════════════════════════════════════════════════════
class MultiTimeframeBuilder:
    def __init__(self):
        self._htf_cache = {}

    def _build_htf_df(self, df_raw, resample_rule):
        if resample_rule in self._htf_cache:
            return self._htf_cache[resample_rule]

        df_htf = (df_raw.set_index('datetime')
                  .resample(resample_rule)
                  .agg({'open':'first','high':'max','low':'min',
                        'close':'last','volume':'sum'})
                  .dropna()
                  .reset_index())

        fe_htf     = FeatureEngineer()
        df_htf_f   = fe_htf.create_features(df_htf, shift=True)
        df_htf_f   = fe_htf.create_target(df_htf_f, horizon=1)

        self._htf_cache[resample_rule] = df_htf_f
        return df_htf_f

    def merge_htf_features(self, df_ltf, df_raw, htf_resample_rule,
                           htf_feature_names, htf_label):
        if not htf_feature_names or htf_resample_rule is None:
            return df_ltf

        df_htf     = self._build_htf_df(df_raw, htf_resample_rule)
        available  = [c for c in htf_feature_names if c in df_htf.columns]

        if not available:
            logging.warning(f"لم يتم إيجاد HTF features لـ {htf_label}")
            return df_ltf

        df_htf_sel = df_htf[['datetime'] + available].copy()
        df_htf_sel.columns = ['datetime'] + [f'htf_{htf_label}_{c}' for c in available]
        df_htf_sel = df_htf_sel.sort_values('datetime')

        # merge_asof: no lookahead — كل LTF تأخذ آخر HTF قبلها فقط
        df_merged  = pd.merge_asof(
            df_ltf.sort_values('datetime'),
            df_htf_sel,
            on='datetime',
            direction='backward'
        )
        df_merged  = df_merged.ffill().fillna(0)
        return df_merged


# ══════════════════════════════════════════════════════════════
# BACKTESTER  (Vectorized — لا predict داخل الـ loop)
# ══════════════════════════════════════════════════════════════
class Backtester:
    def __init__(self, initial_capital=10000, spread_bps=2):
        self.initial_capital = initial_capital
        self.spread_bps      = spread_bps / 10000
        self.slippage        = 0.0001

    def _exec_price(self, price, is_buy):
        if is_buy:
            return price * (1 + self.spread_bps) * (1 + self.slippage)
        return price * (1 - self.spread_bps) * (1 - self.slippage)

    def run(self, df, model, fe, feature_columns, timeframe_config,
            atr_mult_stop=1.5, atr_mult_take=2.0, use_fold_thresholds=False):

        capital, positions, trades = self.initial_capital, [], []
        equity = [capital]

        missing = [c for c in feature_columns + ['close','atr'] if c not in df.columns]
        if missing:
            logging.error(f"أعمدة مفقودة: {missing}"); return {}

        buy_th  = timeframe_config.get(
            'fold_buy_threshold'  if use_fold_thresholds else 'buy_threshold')
        sell_th = timeframe_config.get(
            'fold_sell_threshold' if use_fold_thresholds else 'sell_threshold')

        #  max_trades_per_day: فلتر لمنع الـ overtrading في الـ timeframes الصغيرة
        max_daily    = timeframe_config.get('max_trades_per_day', None)
        daily_counts = {}  # {date_str: count}

        # Vectorized predict — مرة واحدة خارج الـ loop
        X_scaled  = fe.transform(df[feature_columns].copy())
        all_proba = model.predict_proba(X_scaled)[:, 1]
        closes    = df['close'].values
        atrs      = df['atr'].values
        datetimes = df['datetime'].values if 'datetime' in df.columns else None

        for i in range(len(df)):
            p, atr, prob = closes[i], atrs[i], all_proba[i]

            signal = "HOLD"
            if prob > buy_th:    signal = "BUY"
            elif prob < sell_th: signal = "SELL"

            # تطبيق حد الصفقات اليومي — يمنع الـ overtrading في الـ 5min و 15min
            if signal == "BUY" and max_daily and datetimes is not None:
                date_key = str(datetimes[i])[:10]
                if daily_counts.get(date_key, 0) >= max_daily:
                    signal = "HOLD"

            if signal == "BUY" and not positions:
                ep    = self._exec_price(p, True)
                units = capital * 0.1 / ep
                positions.append({
                    'entry_price': ep, 'units': units,
                    'stop_loss':   ep - atr_mult_stop * atr,
                    'take_profit': ep + atr_mult_take * atr
                })
                capital -= units * ep
                # تحديث عداد الصفقات اليومية
                if max_daily and datetimes is not None:
                    date_key = str(datetimes[i])[:10]
                    daily_counts[date_key] = daily_counts.get(date_key, 0) + 1

            elif signal == "SELL" and positions:
                pos = positions.pop(0)
                ep  = self._exec_price(p, False)
                pnl = pos['units'] * (ep - pos['entry_price'])
                capital += pos['units'] * ep
                trades.append({'profit': pnl,
                                'return_pct': pnl / (pos['units'] * pos['entry_price']),
                                'exit_reason': 'signal'})

            if positions:
                pos = positions[0]
                if p <= pos['stop_loss']:
                    ep  = pos['stop_loss']
                    pnl = pos['units'] * (ep - pos['entry_price'])
                    capital += pos['units'] * ep
                    trades.append({'profit': pnl,
                                   'return_pct': pnl / (pos['units'] * pos['entry_price']),
                                   'exit_reason': 'stop_loss'})
                    positions.pop(0)
                elif p >= pos['take_profit']:
                    ep  = pos['take_profit']
                    pnl = pos['units'] * (ep - pos['entry_price'])
                    capital += pos['units'] * ep
                    trades.append({'profit': pnl,
                                   'return_pct': pnl / (pos['units'] * pos['entry_price']),
                                   'exit_reason': 'take_profit'})
                    positions.pop(0)

            equity.append(capital + (positions[0]['units'] * p if positions else 0))

        if positions:
            pos = positions.pop(0)
            ep  = self._exec_price(closes[-1], False)
            pnl = pos['units'] * (ep - pos['entry_price'])
            capital += pos['units'] * ep
            trades.append({'profit': pnl,
                           'return_pct': pnl / (pos['units'] * pos['entry_price']),
                           'exit_reason': 'end_of_data'})

        if not trades: return {}

        eq   = pd.Series(equity)
        wins = [t for t in trades if t['profit'] > 0]
        loss = [t for t in trades if t['profit'] <= 0]
        wr   = len(wins) / len(trades) * 100
        tp   = sum(t['profit'] for t in wins)
        tl   = abs(sum(t['profit'] for t in loss))
        dd   = ((eq - eq.cummax()) / eq.cummax() * 100).min()
        aw   = tp / len(wins) if wins else 0
        al   = tl / len(loss) if loss else 0

        return {
            'total_trades':     len(trades),
            'win_rate':         round(wr, 2),
            'profit_factor':    round(tp / tl, 2) if tl else float('inf'),
            'total_return_pct': round((capital - self.initial_capital) / self.initial_capital * 100, 2),
            'max_drawdown_pct': round(dd, 2),
            'avg_win':          round(aw, 2),
            'avg_loss':         round(al, 2),
            'expectancy':       round((wr/100 * aw) - ((1-wr/100) * al), 2) if al else 0
        }


# ══════════════════════════════════════════════════════════════
# WALK-FORWARD TRAINING  —  مع Multi-Timeframe
# ══════════════════════════════════════════════════════════════
def train_model_with_walk_forward(df_raw, timeframe, n_splits=5,
                                   mtf_builder=None):
    config        = TIMEFRAME_CONFIG[timeframe]
    resample_rule = config['resample_rule']
    years         = config['years']
    min_candles   = config.get('min_candles', 500)
    larger_tf     = config.get('larger_tf')
    htf_features  = config.get('htf_features', [])

    logging.info("=" * 60)
    logging.info(f"Walk-Forward لفريم {timeframe}")
    logging.info("=" * 60)

    # ─── Resample ───
    df_tf = (df_raw.set_index('datetime')
             .resample(resample_rule)
             .agg({'open':'first','high':'max','low':'min','close':'last','volume':'sum'})
             .dropna()
             .reset_index())
    df_tf.rename(columns={'index': 'datetime'}, inplace=True)

    if years:
        cutoff = df_tf['datetime'].max() - timedelta(days=365 * years)
        df_tf  = df_tf[df_tf['datetime'] >= cutoff].copy()
    logging.info(f"شموع بعد القطع: {len(df_tf)}")

    if len(df_tf) < min_candles:
        logging.error(f"بيانات غير كافية ({len(df_tf)} < {min_candles})")
        return None

    # ─── Features (LTF) ───
    fe        = FeatureEngineer()
    df_feat   = fe.create_features(df_tf, shift=True)
    df_target = fe.create_target(df_feat, horizon=1)

    if df_target.empty:
        logging.error("لا توجد بيانات بعد إنشاء الهدف")
        return None

    # ───  Multi-Timeframe: إضافة HTF features ───
    if mtf_builder and larger_tf and htf_features:
        htf_cfg = TIMEFRAME_CONFIG.get(larger_tf)
        if htf_cfg:
            logging.info(f"إضافة HTF features من {larger_tf} ...")
            df_target = mtf_builder.merge_htf_features(
                df_ltf=df_target,
                df_raw=df_raw,
                htf_resample_rule=htf_cfg['resample_rule'],
                htf_feature_names=htf_features,
                htf_label=larger_tf
            )

    # تحديث feature_columns بعد إضافة HTF
    exclude_final = ['datetime','open','high','low','close','volume',
                     'future_close','pct_change','target']
    fe.feature_columns = [c for c in df_target.columns if c not in exclude_final]
    n_features = len(fe.feature_columns)
    logging.info(f"إجمالي الـ features: {n_features}")

    X = df_target[fe.feature_columns]
    y = df_target['target']

    effective_splits = min(n_splits, max(2, len(X) // 100))
    tscv = TimeSeriesSplit(n_splits=effective_splits)

    fold_metrics, best_model, best_score, best_fe_fold = [], None, -np.inf, None

    for fold, (tr_idx, te_idx) in enumerate(tscv.split(X)):
        logging.info(f"Fold {fold+1}/{effective_splits}")
        X_tr, X_te = X.iloc[tr_idx], X.iloc[te_idx]
        y_tr, y_te = y.iloc[tr_idx], y.iloc[te_idx]

        neg = (y_tr == 0).sum(); pos = (y_tr == 1).sum()
        spw = neg / pos if pos > 0 else 1

        fe_fold = FeatureEngineer()
        fe_fold.fit_scaler(X_tr)
        X_tr_s = fe_fold.transform(X_tr)
        X_te_s = fe_fold.transform(X_te)

        model = xgb.XGBClassifier(
            objective='binary:logistic',
            n_estimators=1000, max_depth=7, learning_rate=0.02,
            subsample=0.8, colsample_bytree=0.8,
            reg_alpha=0.1, reg_lambda=1.0,
            scale_pos_weight=spw, random_state=42,
            tree_method='hist', verbosity=0,
            use_label_encoder=False, eval_metric='logloss',
            early_stopping_rounds=50
        )
        model.fit(X_tr_s, y_tr, eval_set=[(X_te_s, y_te)], verbose=False)

        y_pred    = model.predict(X_te_s)
        acc       = accuracy_score(y_te, y_pred)
        precision = precision_score(y_te, y_pred, zero_division=0)
        recall    = recall_score(y_te, y_pred, zero_division=0)

        df_test      = df_target.iloc[te_idx].copy()
        backtester   = Backtester()
        metrics_test = backtester.run(
            df_test, model, fe_fold, fe_fold.feature_columns,
            config, use_fold_thresholds=True
        )

        logging.info(f"  Fold {fold+1} — Acc:{acc:.2%} Prec:{precision:.2%} Rec:{recall:.2%}")
        if metrics_test:
            logging.info(
                f"  BT — WR:{metrics_test['win_rate']}% "
                f"PF:{metrics_test['profit_factor']} "
                f"Ret:{metrics_test['total_return_pct']}% "
                f"Trades:{metrics_test['total_trades']}"
            )

        fold_metrics.append({'fold': fold+1, 'accuracy': acc,
                              'precision': precision, 'recall': recall,
                              'backtest': metrics_test})

        if metrics_test and metrics_test['total_trades'] >= 5:
            score = metrics_test['profit_factor'] * (metrics_test['total_trades'] / 100)
            if score > best_score:
                best_score = score; best_model = model; best_fe_fold = fe_fold

    if best_model is None:
        logging.warning(f"Fallback إلى آخر fold للفريم {timeframe}")
        best_model = model; best_fe_fold = fe_fold

    # ─── Final model على كامل البيانات ───
    neg_f = (y == 0).sum(); pos_f = (y == 1).sum()
    fe_full = FeatureEngineer()
    fe_full.fit_scaler(X)
    X_full_s = fe_full.transform(X)

    final_model = xgb.XGBClassifier(
        objective='binary:logistic',
        n_estimators=best_model.n_estimators,
        max_depth=best_model.max_depth,
        learning_rate=best_model.learning_rate,
        subsample=best_model.subsample,
        colsample_bytree=best_model.colsample_bytree,
        reg_alpha=best_model.reg_alpha,
        reg_lambda=best_model.reg_lambda,
        scale_pos_weight=neg_f / pos_f if pos_f else 1,
        random_state=42, tree_method='hist', verbosity=0,
        use_label_encoder=False, eval_metric='logloss',
    )
    final_model.fit(X_full_s, y)

    os.makedirs("models", exist_ok=True)
    model_path = f"models/{SYMBOL_SAFE}_{timeframe}.pkl"
    joblib.dump({
        'model': final_model, 'scaler': fe_full.scaler,
        'feature_columns': fe_full.feature_columns, 'config': config
    }, model_path)

    # ─── Holdout backtest (آخر 20% — لا data leakage) ───
    holdout_start = int(len(df_target) * 0.8)
    df_holdout    = df_target.iloc[holdout_start:].copy()
    bt_holdout    = Backtester()
    full_metrics  = bt_holdout.run(
        df_holdout, final_model, fe_full,
        fe_full.feature_columns, config, use_fold_thresholds=False
    )

    if full_metrics:
        logging.info(
            f"Holdout (20%الأخيرة) — WR:{full_metrics['win_rate']}% "
            f"PF:{full_metrics['profit_factor']} "
            f"Ret:{full_metrics['total_return_pct']}%"
        )
    else:
        logging.warning(f"لا صفقات في الـ holdout للفريم {timeframe}")

    metrics_file = f"models/{SYMBOL_SAFE}_{timeframe}_metrics.json"
    with open(metrics_file, 'w') as f:
        json.dump({
            'fold_results': fold_metrics,
            'holdout_backtest_last_20pct': full_metrics,
            'config': config
        }, f, indent=2, default=str)

    logging.info(f"تم حفظ النموذج: {model_path}")
    return final_model, fe_full


# ══════════════════════════════════════════════════════════════
# TRAIN ALL MODELS
# ══════════════════════════════════════════════════════════════
def train_all_models(force_refresh_data=False):
    logging.info("=" * 70)
    logging.info("بدء نظام Model Factory — Professional Edition")
    logging.info("=" * 70)

    df_raw = get_or_update_data(CSV_RAW_FILE, target_candles=100000,
                                 force_refresh=force_refresh_data)
    if df_raw.empty:
        logging.error("فشل الحصول على البيانات."); return

    # MTF builder مشترك — يحتفظ بـ cache الـ HTF لتجنب إعادة الحساب
    mtf_builder = MultiTimeframeBuilder()

    results = {}
    for tf in TIMEFRAME_CONFIG:
        logging.info("-" * 50)
        res = train_model_with_walk_forward(
            df_raw, tf, n_splits=5, mtf_builder=mtf_builder
        )
        results[tf] = "✅" if res else "❌"

    logging.info("=" * 70)
    logging.info("ملخص التدريب:")
    for tf, st in results.items():
        logging.info(f"   {tf}: {st}")
    logging.info("=" * 70)


if __name__ == "__main__":
    train_all_models(force_refresh_data=False)

#Download folder

In [ ]:
from google.colab import files
import os, zipfile

# اعمل zip للفولدر كله
with zipfile.ZipFile('models.zip', 'w') as z:
    for f in os.listdir('models'):
        z.write(f'models/{f}')

files.download('models.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>